In [6]:
import pandas as pd
from river import datasets, base, compose, stream, preprocessing, stats
import importlib
from geopy.distance import geodesic
from tqdm.auto import tqdm
from itertools import islice
import numpy as np
import matplotlib.pyplot as plt

import load_model
importlib.reload(load_model)

from load_model import ModelFactory

In [7]:
df = pd.read_csv('dataset/warsaw_osm_network_stream.csv')
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values('timestamp')

df["hour"] = df["timestamp"].dt.hour

df["is_weekend"] = df["is_weekend"].astype(int)
df["rush_hour"] = df["rush_hour"].astype(int)
df["is_holiday"] = df["is_holiday"].astype(int)

categorical_cols = ["weather"]

df = pd.get_dummies(df, columns=categorical_cols, prefix=categorical_cols)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 298253 entries, 0 to 298252
Data columns (total 16 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   timestamp        298253 non-null  datetime64[ns]
 1   start_lat        298253 non-null  float64       
 2   start_lon        298253 non-null  float64       
 3   end_lat          298253 non-null  float64       
 4   end_lon          298253 non-null  float64       
 5   distance_km      298253 non-null  float64       
 6   travel_time_min  298253 non-null  float64       
 7   hour             298253 non-null  int32         
 8   day_of_week      298253 non-null  int64         
 9   is_weekend       298253 non-null  int64         
 10  rush_hour        298253 non-null  int64         
 11  is_holiday       298253 non-null  int64         
 12  weather_clear    298253 non-null  bool          
 13  weather_fog      298253 non-null  bool          
 14  weather_rain     298253 n

In [9]:
# df = df.iloc[:10000]

df.head(10)

,timestamp,start_lat,start_lon,end_lat,end_lon,distance_km,travel_time_min,hour,day_of_week,is_weekend,rush_hour,is_holiday,weather_clear,weather_fog,weather_rain,weather_snow
0,2025-01-01 00:09:06,52.253264,20.981510,52.159729,21.126328,17.786888,25.131083,0,2,0,0,1,True,False,False,False
1,2025-01-01 00:17:26,52.191368,21.043104,52.145980,21.037312,6.231378,11.619191,0,2,0,0,1,False,False,True,False
2,2025-01-01 00:34:09,52.267529,21.032690,52.149947,21.053613,17.694589,55.764620,0,2,0,0,1,False,False,False,True
3,2025-01-01 00:35:16,52.177431,21.062454,52.218237,20.999811,7.952929,19.220696,0,2,0,0,1,False,True,False,False
4,2025-01-01 00:42:18,52.253970,21.220869,52.196762,20.895023,28.631239,42.313451,0,2,0,0,1,False,False,True,False
5,2025-01-01 00:43:43,52.241570,21.083285,52.214244,20.968777,10.235888,17.288726,0,2,0,0,1,True,False,False,False
6,2025-01-01 00:46:36,52.262596,20.914268,52.314712,20.962611,9.597841,21.631542,0,2,0,0,1,False,False,True,False
7,2025-01-01 01:02:15,52.188418,20.927651,52.216389,21.042366,12.126125,50.180485,1,2,0,0,1,False,False,False,True
8,2025-01-01 01:09:55,52.239452,21.028856,52.275270,21.044495,6.171929,17.467685,1,2,0,0,1,False,False,False,True
9,2025-01-01 01:11:19,52.153491,20.994672,52.268914,21.144303,20.732478,26.735606,1,2,0,0,1,True,False,False,False


In [10]:
X = df.drop(columns=["travel_time_min", "timestamp"]).copy()
y = df["travel_time_min"]

categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object", "category"]).columns.tolist()


In [11]:
factory = ModelFactory()
dataset = datasets.Taxis()
model_names = ['ARF', 'HTR', 'SRP']
pipelines = {
    name: factory.create(model_name=name, transformer=None)
    for name in model_names
}

mean_model = stats.Mean()

In [12]:
results = []

for i, (x, target) in tqdm(enumerate(stream.iter_pandas(X, y)), total=len(df), desc="Streaming predictions"):
    row = {
        "ts" : i,
        "actual": target
    }

    y_pred_mean = mean_model.get()
    y_pred_mean = 0.0 if y_pred_mean is None else y_pred_mean

    mean_model.update(target)

    row["MEAN"] = y_pred_mean

    for name, model in pipelines.items():
            
        y_pred = model.predict_one(x)
        y_pred = y_pred if y_pred is not None else 0.0

        model.learn_one(x, target)

        row[name] = y_pred

    results.append(row)

Streaming predictions: 100%|██████████| 298253/298253 [16:22:49<00:00,  5.06it/s]      


In [13]:
res_df = pd.DataFrame(results)
res_df.head(10)

res_df.to_csv('synth_all.csv')